# **Step 1: Load everything from hierarchy**

In [1]:
from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np

drive.mount('/content/drive')
DRIVE     = Path('/content/drive/MyDrive/m5_project')
PROCESSED = DRIVE / 'processed'

%run /content/drive/MyDrive/m5_project/notebooks/hierarchy.ipynb

cols = pd.read_parquet(PROCESSED / 'ca12_long.parquet').columns.tolist()
print(cols)

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(6096944, 8)
days: 847 -> 1941
L1_total            1 series
L2_store            2 series
L3_cat              3 series
L4_dept             7 series
L5_store_cat        6 series
L6_store_dept      14 series
L7_item          3049 series
L8_item_store    6098 series
(6098, 1095)
L1_total            1 series x 1095 days
L2_store            2 series x 1095 days
L3_cat              3 series x 1095 days
L4_dept             7 series x 1095 days
L5_store_cat        6 series x 1095 days
L6_store_dept      14 series x 1095 days
L7_item          3049 series x 1095 days
L8_item_store    6098 series x 1095 days
Level 1: Sum of both the stores
d       847   848   849   850   851
Total  7745  8747  7996  8197  6266
----------------------------------------

 Level 2: Sum of each Store
           847   848   849   850   851
store_id                    

# **Step 2 Calendar table**




In [2]:
cal = (pd.read_parquet(PROCESSED / 'ca12_long.parquet',
                       columns=['d', 'date', 'wday', 'month', 'event_type_1', 'snap_CA'])
         .drop_duplicates('d').set_index('d').sort_index())

cal['day_of_month'] = pd.to_datetime(cal['date']).dt.day
cal['event_type']   = cal['event_type_1'].astype('category').cat.codes      # event kind as a number
cal = cal[['wday', 'month', 'day_of_month', 'event_type', 'snap_CA']]

print(cal.shape)            # expect (1095, 5): one row per day, d_847..d_1941
cal.loc[1914:1920]          # first week of the test period

(1095, 5)


,wday,month,day_of_month,event_type,snap_CA
d,,,,,
1914,3,4,25,-1,0
1915,4,4,26,-1,0
1916,5,4,27,-1,0
1917,6,4,28,-1,0
1918,7,4,29,-1,0
1919,1,4,30,2,0
1920,2,5,1,2,1


In [3]:
print(dict(enumerate(pd.read_parquet(PROCESSED / 'ca12_long.parquet', columns=['event_type_1'])['event_type_1'].astype('category').cat.categories)))

# event codes

{0: 'Cultural', 1: 'National', 2: 'Religious', 3: 'Sporting'}


# **Step 3: Past Sell features**

- The best clue about tomorrow’s sales is past sales. This cell gives each day two kinds of clues:

- lags: what sold exactly x days earlier.
- rolling means: the average over the last y days, counting back from 28 days ago

In [14]:
LAGS    = [28, 35, 42, 49, 56]     # same weekday, 4 to 8 weeks back
WINDOWS = [7, 14, 28, 56]          # average of the last N days, ending 28 days back

def make_features(table, level_name):
  grid = table.copy()
  grid.columns.name = 'd'

  if grid.index.nlevels > 1:               #   ('CA_1','FOODS')
    grid.index = grid.index.map('_'.join)  #   join them -> 'CA_1_FOODS'
  long = grid.rename_axis('series_id').stack().rename('sales').reset_index()
  long['level'] = level_name

  by_series = long.groupby('series_id')['sales']
  for lag in LAGS:
      long[f'lag_{lag}'] = by_series.shift(lag)
  for w in WINDOWS:
      long[f'roll_mean_{w}'] = by_series.transform(lambda s: s.shift(28).rolling(w).mean())
  return long


# **Step 4: Price clues (bottom level only)**
- sell_price: the product's price that day
- price_vs_usual: price ÷ its usual training-period price

In [15]:
p = price.replace(0, np.nan)

usual = p.loc[:, :TRAIN_END].mean(axis=1)          # each product's usual price (training days only)

price_feats = pd.DataFrame({
    'sell_price':     p.stack(),                   # today's price            -> $1.50
    'price_vs_usual': p.div(usual, axis=0).stack() # today / usual price      -> 1.50 / 2.00 = 0.75
}).rename_axis(['series_id', 'd']).reset_index()   # long list: one row per product per day

print(price_feats.shape)
price_feats['price_vs_usual'].describe()

(6096944, 4)


,price_vs_usual
count,6.096944e+06
mean,1.000214e+00
std,3.537119e-02
min,4.000955e-03
25%,9.999998e-01
50%,1.000000e+00
75%,1.001650e+00
max,2.825401e+00


In [16]:
price_feats

# Each row is one product, in one store, on one day, along with its price that day.

,series_id,d,sell_price,price_vs_usual
0,FOODS_1_001_CA_1_evaluation,847,2.24,1.0
1,FOODS_1_001_CA_1_evaluation,848,2.24,1.0
2,FOODS_1_001_CA_1_evaluation,849,2.24,1.0
3,FOODS_1_001_CA_1_evaluation,850,2.24,1.0
4,FOODS_1_001_CA_1_evaluation,851,2.24,1.0
...,...,...,...,...
6096939,HOUSEHOLD_2_516_CA_2_evaluation,1937,5.94,1.0
6096940,HOUSEHOLD_2_516_CA_2_evaluation,1938,5.94,1.0
6096941,HOUSEHOLD_2_516_CA_2_evaluation,1939,5.94,1.0
6096942,HOUSEHOLD_2_516_CA_2_evaluation,1940,5.94,1.0


# **Step 5 — Teacher and student tables**

- Teacher: all series from L1–L4 in one table (13 series) + calendar → one teacher model (paper Sec 5.1.2–5.1.3)
- Student: every item×store (6,098 series) + calendar + price + item/dept/store ids (paper Sec 5.4)


In [20]:
# Teacher: all series from L1-L4 in one table (paper Sec 5.1.2)
teacher = pd.concat([make_features(sales_by_level[name], name) for name in TEACHER_LEVELS],
                    ignore_index=True)
teacher = teacher.join(cal, on='d')
teacher = teacher.dropna(subset=['roll_mean_56'])              # first 83 days: not enough history

# Student: bottom level (item x store)
student = make_features(bottom, 'L8_item_store')
student = (student.merge(price_feats, on=['series_id', 'd'])   # inner join: on-sale days only
                  .join(cal, on='d')
                  .join(meta, on='series_id')                  # ids for store/dept clusters (Sec 5.4)
                  .dropna(subset=['roll_mean_56']))

# float64 -> float32 to save memory
for table in (teacher, student):
    floats = table.select_dtypes('float64').columns
    table[floats] = table[floats].astype('float32')

# leak check: all clues >= 28 days old
assert min(LAGS) >= HORIZON, 'feature too recent -> leakage'

print('teacher:', teacher.shape, '| series:', teacher.series_id.nunique())
print('student:', student.shape, '| series:', student.series_id.nunique())
student.head(10).T


# The table below is Student's table, Where each column is 1 row of the table
### Each row holds everything the model knows about one product on one day + the answer (sales). ###


teacher: (13156, 18) | series: 13
student: (5706588, 24) | series: 6098


,83,84,85,86,87,88,89,90,91,92
series_id,FOODS_1_001_CA_1_evaluation,FOODS_1_001_CA_1_evaluation,FOODS_1_001_CA_1_evaluation,FOODS_1_001_CA_1_evaluation,FOODS_1_001_CA_1_evaluation,FOODS_1_001_CA_1_evaluation,FOODS_1_001_CA_1_evaluation,FOODS_1_001_CA_1_evaluation,FOODS_1_001_CA_1_evaluation,FOODS_1_001_CA_1_evaluation
d,930,931,932,933,934,935,936,937,938,939
sales,1,0,0,1,1,0,0,0,0,0
level,L8_item_store,L8_item_store,L8_item_store,L8_item_store,L8_item_store,L8_item_store,L8_item_store,L8_item_store,L8_item_store,L8_item_store
lag_28,0.0,0.0,1.0,2.0,0.0,1.0,2.0,0.0,0.0,1.0
lag_35,0.0,1.0,2.0,0.0,1.0,3.0,0.0,0.0,0.0,1.0
lag_42,0.0,1.0,1.0,0.0,9.0,2.0,0.0,0.0,1.0,2.0
lag_49,0.0,10.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0
lag_56,0.0,0.0,2.0,1.0,1.0,0.0,0.0,0.0,10.0,0.0
roll_mean_7,1.0,0.857143,0.714286,1.0,0.857143,0.571429,0.857143,0.857143,0.857143,0.857143


In [21]:
teacher.head(10).T

,83,84,85,86,87,88,89,90,91,92
series_id,Total,Total,Total,Total,Total,Total,Total,Total,Total,Total
d,930,931,932,933,934,935,936,937,938,939
sales,6962,8087,10239,9992,7257,6434,7102,6821,7640,10431
level,L1_total,L1_total,L1_total,L1_total,L1_total,L1_total,L1_total,L1_total,L1_total,L1_total
lag_28,6903.0,8006.0,9813.0,10143.0,6504.0,6475.0,6554.0,6674.0,7819.0,9019.0
lag_35,6801.0,8202.0,9320.0,10190.0,6774.0,6592.0,6723.0,6903.0,8006.0,9813.0
lag_42,7673.0,8730.0,8489.0,9825.0,7092.0,6952.0,7030.0,6801.0,8202.0,9320.0
lag_49,6720.0,8526.0,10124.0,9917.0,8143.0,7940.0,8912.0,7673.0,8730.0,8489.0
lag_56,6802.0,7979.0,9341.0,9890.0,6787.0,6488.0,6398.0,6720.0,8526.0,10124.0
roll_mean_7,7814.856934,7786.856934,7857.285645,7850.571289,7812.0,7795.285645,7771.143066,7738.428711,7711.714355,7598.285645


# **Step 6: Finalize results**

In [22]:

assert teacher.shape == (13156, 18),   'teacher table changed! check data or code'
assert student.shape == (5706588, 24), 'student table changed! check data or code'
assert student.series_id.nunique() == 6098
print('features locked ✅')

features locked ✅
